# Cleaning

Runs after `01_data_review.ipynb`. Writes `../data/clean.csv`.

Rule followed here: nothing gets deleted because it looks inconvenient. Rows stay,
columns that are genuinely redundant go, and every null is either given a meaning
or filled with a flag saying it was filled.

In [ ]:
# work on a copy so the raw file is never touched
import pandas as pd, numpy as np

tr = pd.read_csv('../data/train.csv')
df = tr.copy()
len(df)

## Rows: keep all of them

In [ ]:
# the 2.5% with no finish time are drop-outs (see 01, section 4). flag, don't drop.
df['dnf'] = df.actual_finish_time_minutes.isna().astype(int)
df.dnf.sum()

In [ ]:
# no duplicates to worry about
df.runner_id.duplicated().sum(), df.drop(columns='runner_id').duplicated().sum()

## Redundant and useless columns

In [ ]:
# km and miles are the same column, and the km version is the broken one
df = df.drop(columns='weekly_mileage_km')

In [ ]:
# marathon_date: no seasonality at all, so it's not a feature. kept for description only.
pd.to_datetime(tr.marathon_date).dt.month.pipe(
    lambda m: tr.groupby(m).actual_finish_time_minutes.mean().round(1))

## Nulls that mean something

In [ ]:
# no injury recorded -> severity was never going to be filled in
df['injury_severity'] = df.injury_severity.fillna('None')
df.injury_severity.value_counts()

In [ ]:
# personal best is missing exactly for first-timers. say so explicitly.
df['is_first_marathon'] = (df.previous_marathon_count == 0).astype(int)
df.is_first_marathon.mean().round(3)

## Nulls that are just missing

In [ ]:
MCAR = ['vo2_max', 'cross_training_hours_per_week', 'nutrition_score',
        'hydration_consistency', 'sleep_hours_avg']

# check before filling: same missing rate everywhere, same people
for c in MCAR:
    m = df[c].isna()
    print(f'{c:32s} {m.mean():.1%} null, mean age {df.age[m].mean():.1f} vs {df.age[~m].mean():.1f}')

In [ ]:
# nothing predicts the gaps, so median fill is safe. flag it anyway.
for c in MCAR:
    df[c + '_was_missing'] = df[c].isna().astype(int)
    df[c] = df[c].fillna(df[c].median())

df[MCAR].isna().sum().sum()

## Things that are wrong but stay

In [ ]:
# long run longer than the whole training week. 233 rows, impossible, left visible.
bad_long_run = df.long_run_distance_km > df.weekly_mileage_miles * 1.609
df['implausible_volume'] = bad_long_run.astype(int)
bad_long_run.sum()

In [ ]:
# columns sitting on a hard ceiling or a single value. usable, but not for clustering.
print('resting_heart_rate_bpm max:', df.resting_heart_rate_bpm.max())
print('long_run_distance_km = 15.0 in', f'{(df.long_run_distance_km == 15.0).mean():.0%}', 'of rows')
print('training_adherence_pct min:', df.training_adherence_pct.min())

## Derived columns

In [ ]:
# the four medal codes are undefined, so collapse to won something / didn't
df['medal'] = (df.medal_outcome > 0).astype(int)

# how badly the goal was missed. descriptive only, never a predictor.
df['goal_gap'] = df.actual_finish_time_minutes - df.target_finish_time_minutes

df[['medal', 'goal_gap']].describe().round(2)

## Column lists for later

In [ ]:
# personal_best and target time stay out of anything predicting finish time
SPOILERS = ['target_finish_time_minutes', 'personal_best_minutes', 'goal_gap']

# missed_workout_pct is adherence again (r = 0.89), so only one of the two goes in
BEHAVIOUR = ['motivation_level', 'mental_preparation_score', 'training_adherence_pct',
             'consecutive_weeks_no_miss', 'training_streak_days', 'goal_completion_rate',
             'run_club_attendance_rate', 'warmup_adherence_pct', 'stretching_adherence_pct',
             'early_morning_run_frequency', 'weather_condition_training_pct',
             'nutrition_score', 'hydration_consistency', 'sleep_hours_avg', 'recovery_score']

# long_run_distance_km left out: 81% of it is the same number
BODY = ['age', 'running_experience_months', 'previous_marathon_count', 'weekly_mileage_miles',
        'runs_per_week', 'speed_work_sessions_per_week', 'rest_days_per_week',
        'cross_training_hours_per_week', 'resting_heart_rate_bpm', 'vo2_max', 'bmi',
        'injury_count']

len(BEHAVIOUR), len(BODY)

## Save

In [ ]:
# 80k rows in, 80k rows out
df.to_csv('../data/clean.csv', index=False)
df.shape

Same 80,000 rows as the raw file. One column dropped, nine added, no imputation
that cannot be traced back with a flag.